# 02 - Experiments: Vanilla LSTM vs CIFG vs WMC (Drug Review, 3 classes)

All cells are implemented by hand with `nn.Linear` (`src/cells.py`). Every model uses the **same data split**, the same evaluation protocol and the same tuning budget.

**Plan**
1. Smoke test on synthetic data (shapes, equations, masking).
2. Load data and build the fixed split.
3. *Stage 1 - hyper-parameter study* on a stratified fraction of the training set (one factor at a time: learning rate, optimizer, initialisation, batch size, hidden size, class weights; 3 cells).
4. *Stage 2 - final runs*: best configuration of each cell, full training set, several seeds.
5. Tables and figures (saved in `results/`).

Every finished run is appended to `results/metrics.csv`, so if the Kaggle session stops you can simply re-run the notebook: finished runs are skipped.

## 0. Setup

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_NAME = "projet-lstm-drug-review"
REPO_URL = "https://github.com/YOUR_USERNAME/projet-lstm-drug-review.git"  # <-- edit after publishing

ON_KAGGLE = Path("/kaggle/input").exists()


def locate_repo() -> Path:
    """Find the project code: current folder, uploaded Kaggle dataset, or GitHub clone."""
    # 1) already inside the repository (local machine, or already cloned)
    for folder in (Path.cwd(), Path.cwd().parent, Path.cwd() / REPO_NAME):
        if (folder / "src" / "train.py").is_file():
            return folder
    # 2) code uploaded as a Kaggle dataset (folder containing src/train.py)
    if ON_KAGGLE:
        for train_py in Path("/kaggle/input").rglob("train.py"):
            if train_py.parent.name == "src":
                dest = Path("/kaggle/working") / REPO_NAME
                if not dest.exists():
                    shutil.copytree(train_py.parent.parent, dest)
                return dest
    # 3) clone from GitHub (needs Internet enabled in the Kaggle notebook settings)
    dest = (Path("/kaggle/working") if ON_KAGGLE else Path.cwd()) / REPO_NAME
    if not dest.exists():
        subprocess.run(["git", "clone", REPO_URL, str(dest)], check=True)
    return dest


REPO = locate_repo()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Repository:", REPO)

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

from src.cells import CELL_LABELS
from src.data import DataConfig, build_bundle
from src.train import (ExperimentConfig, config_from_row, get_device, load_metrics,
                       load_run_details, plot_confusion_matrix, plot_hparam_effect,
                       plot_model_comparison, plot_training_curves, run_experiment,
                       smoke_test)

device = get_device()
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU (enable it in Kaggle: Settings > Accelerator)")

# ----------------------------- budget settings -----------------------------
QUICK_MODE = False          # True = tiny run just to check everything works end-to-end (separate results folder)
CELLS = ["lstm", "cifg", "wmc"]

if QUICK_MODE:
    STUDY_FRACTION, EPOCHS_STUDY, EPOCHS_FINAL, FINAL_SEEDS = 0.02, 1, 1, [0]
else:
    STUDY_FRACTION = 0.2    # stratified share of the training split used in the hyper-parameter study
    EPOCHS_STUDY = 5
    EPOCHS_FINAL = 8
    FINAL_SEEDS = [0, 1]
STUDY_SEEDS_BASE = [0, 1]   # the default configuration is run with 2 seeds to measure seed noise

RESULTS_DIR = str(REPO / ("results_quick" if QUICK_MODE else "results"))
FIG_DIR = Path(RESULTS_DIR) / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Results folder:", RESULTS_DIR)

## 1. Smoke test (synthetic data, ~1 minute)
Checks: cells match the equations, no `nn.LSTM`, padding does not change outputs, a full train/eval/save cycle works for the three cells. **It must print `Smoke test passed.`**

In [ ]:
smoke_test(device)

## 2. Data
One fixed split (seed 42): official test file, stratified 10% validation from the official train file, vocabulary from the training split only.

In [ ]:
bundle = build_bundle(DataConfig(label_edges=(4, 6), max_len=200, min_freq=2, max_vocab=30000, split_seed=42))
print(bundle.summary())

## 3. Stage 1 - hyper-parameter study (one factor at a time)

Default configuration: Adam, lr 1e-3, Xavier init, batch size 64, hidden size 128, embedding 128, dropout 0.3, last-state pooling, no class weights.
Each axis changes **one** setting; the three cells get exactly the same grid. SGD uses momentum 0.9 and a larger learning rate (0.1), RMSprop and Adam use 1e-3. The default configuration is also run with a second seed to see how large the seed-to-seed noise is.

In [ ]:
BASE = dict(embed_dim=128, hidden_size=128, num_layers=1, bidirectional=False, dropout=0.3,
            pooling="last", optimizer="adam", lr=1e-3, init="xavier", batch_size=64,
            use_class_weights=False, epochs=EPOCHS_STUDY, patience=2,
            train_fraction=STUDY_FRACTION, group="hp_study")

AXES = {
    "lr": [3e-4, 1e-3, 3e-3],
    "optimizer": [("adam", 1e-3), ("rmsprop", 1e-3), ("sgd", 0.1)],   # (optimizer, learning rate)
    "init": ["xavier", "orthogonal", "default"],
    "batch_size": [32, 64, 128],
    "hidden_size": [64, 128, 256],
    "use_class_weights": [False, True],
}


def make_study_configs():
    """All Stage-1 runs, ordered so that the three cells are always run side by side."""
    configs = {}
    for seed in STUDY_SEEDS_BASE:
        for cell in CELLS:
            cfg = ExperimentConfig(cell=cell, seed=seed, **BASE)
            configs[cfg.run_id] = cfg
    for axis, values in AXES.items():
        for value in values:
            override = dict(zip(("optimizer", "lr"), value)) if axis == "optimizer" else {axis: value}
            for cell in CELLS:
                cfg = ExperimentConfig(cell=cell, seed=0, **{**BASE, **override})
                configs.setdefault(cfg.run_id, cfg)       # the default config appears once
    return list(configs.values())


study_configs = make_study_configs()
print(len(study_configs), "runs in Stage 1")

In [ ]:
start = time.time()
for k, cfg in enumerate(study_configs, 1):
    print(f"\n=== Stage 1: run {k}/{len(study_configs)} ===")
    run_experiment(cfg, bundle, device, RESULTS_DIR, save_figures=False)
print(f"\nStage 1 finished in {(time.time() - start) / 60:.1f} minutes")

### Stage 1 results
Selection uses the **validation** set only. Test numbers are not used to choose anything.

In [ ]:
metrics = load_metrics(RESULTS_DIR)
study = metrics[(metrics["group"] == "hp_study") & np.isclose(metrics["train_fraction"], STUDY_FRACTION)]
print(len(study), "Stage-1 runs found")

# seed noise: default configuration, seeds 0 and 1
default_mask = ((study["optimizer"] == BASE["optimizer"]) & np.isclose(study["lr"], BASE["lr"]) &
                (study["init"] == BASE["init"]) & (study["batch_size"] == BASE["batch_size"]) &
                (study["hidden_size"] == BASE["hidden_size"]) & (~study["use_class_weights"]))
display(study[default_mask].pivot_table(index="cell", columns="seed",
                                        values=["val_accuracy", "val_macro_f1"]).round(4))

In [ ]:
AXIS_COLS = ["lr", "optimizer", "init", "batch_size", "hidden_size", "use_class_weights"]
OPT_PAIRS = set(AXES["optimizer"])


def axis_slice(df, axis):
    """Rows where only `axis` differs from the default configuration (seed 0)."""
    mask = df["seed"] == 0
    free = {"lr"} if axis == "optimizer" else set()    # each optimizer has its own learning rate
    for col in AXIS_COLS:
        if col != axis and col not in free:
            mask &= df[col] == BASE[col]
    out = df[mask]
    if axis == "optimizer":
        out = out[[(o, l) in OPT_PAIRS for o, l in zip(out["optimizer"], out["lr"])]]
    return out


for axis in AXES:
    part = axis_slice(study, axis)
    print(f"\n##### {axis}")
    display(part.pivot_table(index=axis, columns="cell",
                             values=["val_macro_f1", "val_accuracy", "train_time_s"]).round(4))
    fig = plot_hparam_effect(part, axis, FIG_DIR / f"hp_{axis}.png", metric="val_macro_f1")
    plt.show()

### Select the best configuration of each cell (highest validation macro-F1, seed 0)

In [ ]:
seed0 = study[study["seed"] == 0]
best_rows = (seed0.sort_values("val_macro_f1", ascending=False).groupby("cell").head(1)
             .set_index("cell").reindex(CELLS).reset_index())
display(best_rows[["cell", "optimizer", "lr", "init", "batch_size", "hidden_size",
                   "use_class_weights", "val_macro_f1", "val_accuracy", "trainable_params"]])

## 4. Stage 2 - final runs on the full training set
The best configuration of each cell is retrained on the **full** training split with several seeds (more epochs, patience 3). The test set is evaluated once per run, with the weights of the best validation epoch.

In [ ]:
final_configs = []
for seed in FINAL_SEEDS:
    for _, row in best_rows.iterrows():
        final_configs.append(config_from_row(row.to_dict(), group="final", seed=seed,
                                             train_fraction=1.0, epochs=EPOCHS_FINAL, patience=3))
print(len(final_configs), "final runs")

start = time.time()
for k, cfg in enumerate(final_configs, 1):
    print(f"\n=== Stage 2: run {k}/{len(final_configs)} ===")
    run_experiment(cfg, bundle, device, RESULTS_DIR, save_figures=True)
print(f"\nStage 2 finished in {(time.time() - start) / 60:.1f} minutes")

## 5. Final comparison

In [ ]:
metrics = load_metrics(RESULTS_DIR)
final = metrics[metrics["group"] == "final"]

summary = final.groupby("cell").agg(
    n_runs=("seed", "count"),
    test_accuracy_mean=("test_accuracy", "mean"), test_accuracy_std=("test_accuracy", "std"),
    test_macro_f1_mean=("test_macro_f1", "mean"), test_macro_f1_std=("test_macro_f1", "std"),
    trainable_params=("trainable_params", "first"), recurrent_params=("recurrent_params", "first"),
    train_time_s_mean=("train_time_s", "mean"), best_epoch_mean=("best_epoch", "mean"),
).reindex(CELLS)
display(summary.round(4))
summary.to_csv(Path(RESULTS_DIR) / "summary_final.csv")

class_cols = [c for c in final.columns if c.startswith("test_f1_")]
print("Per-class test F1 (mean over seeds):")
display(final.groupby("cell")[class_cols].mean().reindex(CELLS).round(4))

fig = plot_model_comparison(final, FIG_DIR / "model_comparison.png")
plt.show()

In [ ]:
for cell in CELLS:
    row = final[(final["cell"] == cell) & (final["seed"] == FINAL_SEEDS[0])].iloc[0]
    details = load_run_details(row["run_id"], RESULTS_DIR)
    label = CELL_LABELS[cell]
    plot_confusion_matrix(details["test"]["confusion_matrix"], details["class_names"],
                          FIG_DIR / f"confusion_matrix_{cell}.png", title=f"{label} - test confusion matrix")
    plt.show()
    plot_training_curves(details["history"], FIG_DIR / f"training_curves_{cell}.png",
                         title=f"{label} - training curves (seed {FINAL_SEEDS[0]})")
    plt.show()

## 6. Export the results
On Kaggle, everything written in `/kaggle/working` is kept in the notebook output (use *Save Version* > *Save & Run All*). This cell also makes a zip you can download.

In [ ]:
import shutil
zip_base = (Path("/kaggle/working") if ON_KAGGLE else REPO) / "results_export"
shutil.make_archive(str(zip_base), "zip", RESULTS_DIR)
print("Created", str(zip_base) + ".zip")
print("Files in results/:", sorted(p.name for p in Path(RESULTS_DIR).iterdir()))

## 7. Analysis notes (fill in after the runs)
- Which cell is best on test macro-F1 and accuracy? Is the gap larger than the seed-to-seed std?
- Parameter counts: CIFG has about 3/4 of the vanilla recurrent parameters; WMC adds 3 x H x H. Does performance follow complexity?
- Which hyper-parameters mattered most (see the Stage-1 plots)? Is the effect the same for the three cells?
- Effect of class weights on macro-F1 and on the minority class F1.
- Where do the models fail (confusion matrices, neutral class)? Ideas for improvement: bidirectional / stacked layers, mean or max pooling, pre-trained embeddings, longer training.